# Case-01：一次靜不定梁 (Propped Cantilever Beam)

**課程第一步**——slope_deflection_framework 專案的起點。

本版直接從 GitHub 抓取最新的 `sd_framework.py` / `model_propped_cantilever.py`，不在 notebook 裡內嵌原始碼，避免兩份拷貝分岔的問題。

## 題目摘要
| 項目 | 內容 |
|---|---|
| 結構 | 單跨梁，A端固定、B端滾支承 |
| 跨度 | L = 6.0 m |
| 載重 | 均佈載重 w = 20.0 kN/m (向下) |
| 靜不定度 | 1 (只有一個未知量 θ_B) |
| 邊界條件 | θ_A=0 (固定端不轉動)，M_BA=0 (滾支承不傳彎矩) |

## 驗證方式
每個 Case 都要輸出**結構受力圖 → 剪力圖(SFD) → 彎矩圖(BMD)** 三張圖，並且跟 anastruct 逐一比對，額外做剪力交叉檢查。

## 0. 安裝套件 + 抓取最新原始碼

In [ ]:
try:
    import anastruct
    print("已偵測到 anastruct，略過安裝")
except ImportError:
    print("未偵測到 anastruct，開始安裝...")
    !pip install anastruct -q --break-system-packages
    try:
        import anastruct
        print("安裝完成，可以繼續往下執行")
    except ImportError:
        print("安裝後仍無法載入，這是 Colab 偶爾會發生的快取問題，"
              "請直接重新執行這個 cell 一次（通常第二次就會成功）")

%matplotlib inline

In [ ]:
# 從 GitHub 抓取最新版本的框架與模型原始碼 (main分支)
# 注意: 這一步需要網路連線, 且抓到的是"當下 main 分支的最新內容"，
# 不是寫死在 notebook 裡的某個固定版本
!wget -q -O sd_framework.py https://raw.githubusercontent.com/zhixiu0223/slope_deflection_framework/main/sd_framework.py
!wget -q -O model_propped_cantilever.py https://raw.githubusercontent.com/zhixiu0223/slope_deflection_framework/main/samples/model_propped_cantilever.py

from sd_framework import SlopeDeflectionSolver
from model_propped_cantilever import PropChedCantileverProblem
print("原始碼抓取完成")

## 1. 求解並產生「手寫詳解」風格輸出

結構受力圖 → 剪力圖(SFD) → 彎矩圖(BMD)

In [ ]:
problem = PropChedCantileverProblem(L=6.0, w=20.0)
SlopeDeflectionSolver(problem).solve_and_report()

## 2. anastruct 獨立建模驗證

In [ ]:
from anastruct import SystemElements
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

L, w = 6.0, 20.0
ss = SystemElements(EA=1e8, EI=15000.0)
ss.add_element(location=[[0, 0], [L, 0]])
ss.add_support_fixed(node_id=1)
ss.add_support_roll(node_id=2, direction='x')
ss.q_load(element_id=1, q=-w)
ss.solve()

print("A端反力:", ss.get_node_results_system(1))
print("B端反力:", ss.get_node_results_system(2))

ss.show_structure(figsize=(8, 3))
ss.show_shear_force(figsize=(8, 4))

fig = ss.show_bending_moment(show=False, figsize=(8, 4))
ax = fig.axes[0]
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.grid(True, linestyle='--', alpha=0.4)
ax.axvline(0, color='gray', lw=0.6, zorder=0)
ax.axvline(L, color='gray', lw=0.6, zorder=0)
plt.show()

## 3. 剪力交叉檢查

In [ ]:
Mi, Mj = 90.0, 0.0
C1 = (Mj - Mi - 0.5*w*L**2) / L
V0_predict, VL_predict = C1, C1 + w*L
print(f"我們框架推算剪力: V(0)={V0_predict:.3f}  V(L)={VL_predict:.3f}")

el = ss.element_map[1]
print(f"anastruct 實際剪力: V(0)={el.shear_force[0]:.3f}  V(L)={el.shear_force[-1]:.3f}")

assert abs(V0_predict - el.shear_force[0]) < 0.01, "剪力對不上!"
assert abs(VL_predict - el.shear_force[-1]) < 0.01, "剪力對不上!"
print("\n剪力交叉檢查通過 ✓")